# ஆலமரத்துல விளையாட்டு — AI-Animated Video Generation (Open-Source Text-to-Video)

This notebook implements the **AI Video Generation Pipeline** from the Executive Architecture
Design Document, but replaces the static Ken-Burns pans over the textbook illustration with
real AI-generated animation, using an **open-source** text-to-video diffusion model
(`Wan-AI/Wan2.1-T2V-1.3B-Diffusers`, Apache-2.0 license) running on a free Colab GPU.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

**Expectation setting:** Wan2.1-1.3B is meaningfully better at coherent motion and prompt-following
than older small open models (e.g. damo-vilab's 2023 model), and generates at 480p instead of
256p — but it's still a 1.3B-parameter model on a free GPU, not a commercial-grade video model.
Expect clean, simple motion (an animal nodding, swaying, dancing) to render well; anything with
fine articulated detail (individual finger movement) will still be approximate. Pipeline stages
below mirror the design doc: Storyboard (prompts) -> Video Rendering (AI clips) -> Narration (TTS)
-> Assembly (composite + captions + mux).


## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# Fail fast if there's no GPU, rather than discovering it after a 20GB download.
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected in this Colab runtime. Go to Runtime > Change runtime type > "
        "select a GPU (T4) > Save, then Runtime > Restart session, then run all cells again. "
        "If you already selected a GPU and still see this, you've likely hit Colab's "
        "free-tier GPU quota for now — wait a while and retry, or use Colab Pro."
    )
print("GPU OK:", torch.cuda.get_device_name(0))


In [ ]:
# Colab preinstalls MoviePy 1.x, where top-level `from moviepy import AudioFileClip` doesn't
# exist (that only works in MoviePy 2.x). `-U` forces the upgrade instead of a silent no-op.
!pip install -q -U diffusers transformers accelerate bitsandbytes imageio imageio-ffmpeg ftfy \
    edge-tts "moviepy>=2.0" uharfbuzz freetype-py


In [ ]:
import moviepy
from packaging.version import Version

if Version(moviepy.__version__) < Version("2.0"):
    raise RuntimeError(
        f"moviepy is still {moviepy.__version__} (need >=2.0). The pip install above "
        "upgraded the package on disk, but this kernel already had the old one loaded. "
        "Go to Runtime > Restart session (not Restart runtime and run all — just restart), "
        "then re-run every cell from the top. Do NOT re-run only the pip install cell alone."
    )
print("moviepy", moviepy.__version__, "OK")


In [ ]:
# Tamil-capable font (Colab has no Nirmala UI, so pull Noto Sans Tamil from Google Fonts)
!wget -q -O NotoSansTamil.ttf \
    "https://github.com/google/fonts/raw/main/ofl/notosanstamil/NotoSansTamil%5Bwdth%2Cwght%5D.ttf"
!ls -la NotoSansTamil.ttf


## 2. Tamil text rendering (HarfBuzz + FreeType)

Pillow's default text layout doesn't run OpenType shaping, which breaks Tamil vowel signs that
visually reorder around the base consonant. This shapes with HarfBuzz first, then rasterizes each
positioned glyph with FreeType, same approach used for the static-image version of this pipeline.


In [ ]:
import uharfbuzz as hb
import freetype
from PIL import Image, ImageDraw
import numpy as np

FONT_PATH = "NotoSansTamil.ttf"


def _load_face(font_path=FONT_PATH, font_size=64, index=0):
    face = freetype.Face(font_path, index=index)
    face.set_char_size(font_size * 64)
    return face


def _load_hb_font(font_path=FONT_PATH, font_size=64, index=0):
    blob = hb.Blob.from_file_path(font_path)
    face = hb.Face(blob, index)
    font = hb.Font(face)
    font.scale = (font_size * 64, font_size * 64)
    return font


def shape_text(text, font_size=64, font_path=FONT_PATH, index=0):
    buf = hb.Buffer()
    buf.add_str(text)
    buf.guess_segment_properties()
    font = _load_hb_font(font_path, font_size, index)
    hb.shape(font, buf, {"kern": True, "liga": True})
    return buf.glyph_infos, buf.glyph_positions


def render_text(text, font_size=64, color=(0, 0, 0, 255), font_path=FONT_PATH, index=0, padding=8):
    infos, positions = shape_text(text, font_size, font_path, index)
    ft_face = _load_face(font_path, font_size, index)

    pen_x, pen_y = 0, 0
    glyph_boxes = []
    for info, pos in zip(infos, positions):
        ft_face.load_glyph(info.codepoint, freetype.FT_LOAD_RENDER)
        bitmap = ft_face.glyph.bitmap
        left = ft_face.glyph.bitmap_left
        top = ft_face.glyph.bitmap_top
        w, h, pitch = bitmap.width, bitmap.rows, bitmap.pitch
        if w and h:
            buf = np.array(bitmap.buffer, dtype=np.uint8).reshape(h, abs(pitch))[:, :w].copy()
        else:
            buf = None
        x = pen_x + (pos.x_offset / 64)
        y = pen_y - (pos.y_offset / 64)
        glyph_boxes.append((x + left, y - top, w, h, buf, left, top))
        pen_x += pos.x_advance / 64
        pen_y += pos.y_advance / 64

    if not glyph_boxes:
        return Image.new("RGBA", (1, 1), (0, 0, 0, 0))

    min_x = min(b[0] for b in glyph_boxes)
    max_x = max(b[0] + b[2] for b in glyph_boxes)
    min_y = min(b[1] for b in glyph_boxes)
    max_y = max(b[1] + b[3] for b in glyph_boxes)

    width = int(max_x - min_x) + padding * 2 + 2
    height = int(max_y - min_y) + padding * 2 + 2
    canvas = np.zeros((height, width, 4), dtype=np.uint8)

    for x, y, w, h, buf, left, top in glyph_boxes:
        if w == 0 or h == 0 or buf is None:
            continue
        ox = int(x - min_x) + padding
        oy = int(y - min_y) + padding
        for row in range(h):
            ty = oy + row
            if ty < 0 or ty >= height:
                continue
            for col in range(w):
                tx = ox + col
                if tx < 0 or tx >= width:
                    continue
                alpha = buf[row, col]
                if alpha == 0:
                    continue
                canvas[ty, tx] = [color[0], color[1], color[2], alpha]

    return Image.fromarray(canvas, "RGBA")


def wrap_tamil(text, font_size, max_width):
    words = text.split(" ")
    lines, current = [], ""
    for w in words:
        trial = (current + " " + w).strip()
        img = render_text(trial, font_size=font_size, color=(0, 0, 0, 255))
        if img.width > max_width and current:
            lines.append(current)
            current = w
        else:
            current = trial
    if current:
        lines.append(current)
    return lines


TARGET_W, TARGET_H = 1280, 720


def caption_bar(text, font_size=42, max_width=1120, fg=(30, 20, 10, 255)):
    lines = wrap_tamil(text, font_size, max_width)
    line_imgs = [render_text(l, font_size=font_size, color=fg) for l in lines]
    line_h = max(im.height for im in line_imgs)
    pad_v, pad_h = 22, 34
    bar_h = pad_v * 2 + line_h * len(line_imgs) + 10 * (len(line_imgs) - 1)
    bar_w = TARGET_W - 80

    bar = Image.new("RGBA", (bar_w, bar_h), (0, 0, 0, 0))
    draw = ImageDraw.Draw(bar)
    draw.rounded_rectangle([0, 0, bar_w, bar_h], radius=24, fill=(255, 250, 235, 235))

    y = pad_v
    for im in line_imgs:
        x = (bar_w - im.width) // 2
        bar.paste(im, (x, y), im)
        y += im.height + 10

    canvas = Image.new("RGBA", (TARGET_W, TARGET_H), (0, 0, 0, 0))
    canvas.paste(bar, (40, TARGET_H - bar_h - 36), bar)
    return canvas


def title_banner(text, font_size=40, fg=(255, 255, 255, 255)):
    img = render_text(text, font_size=font_size, color=fg)
    pad_h, pad_v = 30, 14
    bar = Image.new("RGBA", (img.width + pad_h * 2, img.height + pad_v * 2), (0, 0, 0, 0))
    draw = ImageDraw.Draw(bar)
    draw.rounded_rectangle([0, 0, bar.width, bar.height], radius=18, fill=(20, 90, 60, 210))
    bar.paste(img, (pad_h, pad_v), img)
    canvas = Image.new("RGBA", (TARGET_W, TARGET_H), (0, 0, 0, 0))
    canvas.paste(bar, ((TARGET_W - bar.width) // 2, 34), bar)
    return canvas


# sanity check
render_text("கொஞ்சும் கிளியே தலையாட்டு", font_size=50, color=(0, 0, 0, 255))
print("Tamil shaping OK")


## 3. Storyboard

Same beats as the design doc's shared `VideoGenerationState`, but each beat now carries an
**English visual prompt** for the open-source text-to-video model (these models are trained on
English captions) instead of a crop box into the textbook illustration.


In [ ]:
# Each beat = one narration audio track + one-or-more AI video sub-shots that share it
# (mirrors the "shots" list in the static-image version's storyboard.json).
STORY = {
    "chapter_title": "\u0b86\u0bb2\u0bae\u0bb0\u0ba4\u0bcd\u0ba4\u0bc1\u0bb2 \u0bb5\u0bbf\u0bb3\u0bc8\u0baf\u0bbe\u0b9f\u0bcd\u0b9f\u0bc1",
    "voice": "ta-IN-PallaviNeural",
    "beats": [
        {"id": "intro", "narration": "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd \u0b95\u0bc1\u0bb4\u0ba8\u0bcd\u0ba4\u0bc8\u0b95\u0bb3\u0bc7! \u0b87\u0ba9\u0bcd\u0bb1\u0bc1 \u0ba8\u0bbe\u0bae\u0bcd \u0b86\u0bb2\u0bae\u0bb0\u0ba4\u0bcd\u0ba4\u0b9f\u0bbf\u0baf\u0bbf\u0bb2\u0bcd \u0bb5\u0bbf\u0bb3\u0bc8\u0baf\u0bbe\u0b9f\u0bc1\u0bae\u0bcd \u0b95\u0bc1\u0b9f\u0bcd\u0b9f\u0bbf \u0bb5\u0bbf\u0bb2\u0b99\u0bcd\u0b95\u0bc1\u0b95\u0bb3\u0bbf\u0ba9\u0bcd \u0baa\u0bbe\u0b9f\u0bcd\u0b9f\u0bc8\u0b95\u0bcd \u0b95\u0bc7\u0b9f\u0bcd\u0baa\u0bcb\u0bae\u0bcd. \u0bb5\u0bbe\u0bb0\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd, \u0ba8\u0bbe\u0bae\u0bc1\u0bae\u0bcd \u0b95\u0bc8 \u0ba4\u0b9f\u0bcd\u0b9f\u0bbf \u0baa\u0bbe\u0b9f\u0bb2\u0bbe\u0bae\u0bcd!",
         "clips": [{"prompt": "wide shot of a lush green banyan tree in a colorful cartoon forest, warm sunny children's storybook illustration, gentle breeze moving the leaves", "frames": 25, "hold": 1.0}]},
        {"id": "stanza1", "narration": "\u0b86\u0bb2\u0bae\u0bb0\u0ba4\u0bcd\u0ba4\u0bc1\u0bb2 \u0bb5\u0bbf\u0bb3\u0bc8\u0baf\u0bbe\u0b9f\u0bcd\u0b9f\u0bc1, \u0b85\u0ba3\u0bbf\u0bb2\u0bc7 \u0b85\u0ba3\u0bbf\u0bb2\u0bc7 \u0b95\u0bc8\u0ba4\u0b9f\u0bcd\u0b9f\u0bc1",
         "clips": [{"prompt": "a cute cartoon squirrel clapping its tiny hands happily on a tree branch, colorful children's book illustration style", "frames": 21, "hold": 1.0}]},
        {"id": "stanza2", "narration": "\u0b95\u0bc1\u0b95\u0bcd\u0b95\u0bc2 \u0b95\u0bc1\u0b95\u0bcd\u0b95\u0bc2 \u0b95\u0bc1\u0baf\u0bbf\u0bb2\u0bcd\u0baa\u0bbe\u0b9f\u0bcd\u0b9f\u0bc1, \u0b95\u0bca\u0b9e\u0bcd\u0b9a\u0bc1\u0bae\u0bcd \u0b95\u0bbf\u0bb3\u0bbf\u0baf\u0bc7 \u0ba4\u0bb2\u0bc8\u0baf\u0bbe\u0b9f\u0bcd\u0b9f\u0bc1",
         "clips": [{"prompt": "a cheerful green parrot nodding its head while a small dark cuckoo bird sings, sitting on a tree branch, colorful cartoon illustration", "frames": 21, "hold": 1.0}]},
        {"id": "stanza3", "narration": "\u0b95\u0bc1\u0b9f\u0bcd\u0b9f\u0bbf\u0b95\u0bcd\u0b95\u0bc1\u0bb0\u0b99\u0bcd\u0b95\u0bc7 \u0bb5\u0bbe\u0bb2\u0bbe\u0b9f\u0bcd\u0b9f\u0bc1, \u0b95\u0bc1\u0bb3\u0bcd\u0bb3 \u0ba8\u0bb0\u0bbf\u0baf\u0bc7 \u0ba4\u0bbe\u0bb2\u0bbe\u0b9f\u0bcd\u0b9f\u0bc1",
         "clips": [
             {"prompt": "a playful baby monkey hanging from a jungle vine, wagging its tail, cute cartoon children's book illustration", "frames": 17, "hold": 0.5},
             {"prompt": "a small orange fox gently rocking side to side as if singing a lullaby, cute cartoon illustration style", "frames": 17, "hold": 0.5},
         ]},
        {"id": "stanza4", "narration": "\u0b9a\u0bbf\u0ba9\u0bcd\u0ba9 \u0bae\u0bc1\u0baf\u0bb2\u0bc7 \u0bae\u0bc7\u0bb3\u0b99\u0bcd\u0b95\u0bca\u0b9f\u0bcd\u0b9f\u0bc1, \u0b9a\u0bbf\u0b99\u0bcd\u0b95\u0b95\u0bcd\u0b95\u0bc1\u0b9f\u0bcd\u0b9f\u0bbf\u0baf\u0bc7 \u0ba4\u0bbe\u0bb3\u0ba8\u0bcd\u0ba4\u0b9f\u0bcd\u0b9f\u0bc1",
         "clips": [{"prompt": "a small rabbit happily beating a toy drum next to a smiling lion cub clapping along, colorful cartoon illustration", "frames": 21, "hold": 1.0}]},
        {"id": "stanza5", "narration": "\u0b8e\u0bb2\u0bcd\u0bb2\u0bbe\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bbe\u0ba9\u0bcd \u0b86\u0b9f\u0bbf\u0b95\u0bcd\u0b95\u0bbf\u0b9f\u0bcd\u0b9f\u0bc1, \u0b8f\u0bb2\u0bc7\u0bb2\u0bc7\u0bb2\u0bc7\u0bb2\u0bcb \u0baa\u0bbe\u0b9f\u0bbf\u0b95\u0bcd\u0b95\u0bbf\u0b9f\u0bcd\u0b9f\u0bc1",
         "clips": [{"prompt": "many cute cartoon forest animals dancing together happily under a tree, colorful children's book illustration, joyful scene", "frames": 25, "hold": 1.0}]},
        {"id": "stanza6", "narration": "\u0b92\u0ba9\u0bcd\u0bb1\u0bbe\u0b95\u0ba4\u0bcd\u0ba4\u0bbe\u0ba9\u0bcd \u0b9a\u0bc7\u0bb0\u0bcd\u0ba8\u0bcd\u0ba4\u0bc1\u0b95\u0bbf\u0b9f\u0bcd\u0b9f\u0bc1, \u0b93\u0b9f\u0bbf\u0bb5\u0bbe\u0b99\u0bcd\u0b95 \u0ba4\u0bc1\u0bb3\u0bcd\u0bb3\u0bbf\u0b95\u0bcd\u0b95\u0bbf\u0b9f\u0bcd\u0b9f\u0bc1",
         "clips": [{"prompt": "cartoon forest animals running and jumping together on green grass under a big tree, joyful motion, children's book style", "frames": 25, "hold": 1.0}]},
        {"id": "outro", "narration": "\u0b8e\u0bb2\u0bcd\u0bb2\u0bcb\u0bb0\u0bc1\u0bae\u0bcd \u0b9a\u0bc7\u0bb0\u0bcd\u0ba8\u0bcd\u0ba4\u0bc1 \u0baa\u0bbe\u0b9f\u0bb2\u0bc8 \u0baa\u0bbe\u0b9f\u0bbf, \u0b95\u0bc8 \u0ba4\u0b9f\u0bcd\u0b9f\u0bbf \u0bae\u0b95\u0bbf\u0bb4\u0bcd\u0b9a\u0bcd\u0b9a\u0bbf\u0baf\u0bbe\u0b95 \u0b86\u0b9f\u0bbf\u0ba9\u0bcb\u0bae\u0bcd \u0b85\u0bb2\u0bcd\u0bb2\u0bb5\u0bbe! \u0b87\u0ba4\u0bcb \u0bae\u0bc1\u0b9f\u0bbf\u0ba8\u0bcd\u0ba4\u0ba4\u0bc1 \u0ba8\u0bae\u0bcd\u0bae\u0bc1\u0b9f\u0bc8\u0baf \u0b86\u0bb2\u0bae\u0bb0\u0baa\u0bcd \u0baa\u0bbe\u0b9f\u0bcd\u0b9f\u0bc1. \u0bae\u0bc0\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd \u0b9a\u0ba8\u0bcd\u0ba4\u0bbf\u0baa\u0bcd\u0baa\u0bcb\u0bae\u0bcd!",
         "clips": [{"prompt": "cute cartoon forest animals waving goodbye under a big banyan tree, warm children's storybook illustration, sunset light", "frames": 25, "hold": 1.0}]},
    ],
}
print(len(STORY["beats"]), "beats,", sum(len(b["clips"]) for b in STORY["beats"]), "AI clips total")


## 4. Load the open-source text-to-video model

`Wan-AI/Wan2.1-T2V-1.3B-Diffusers` (Apache-2.0). The VAE is kept in fp32 (Wan's recommendation —
it's numerically sensitive in fp16/bf16), the diffusion transformer runs in bf16, and
`enable_model_cpu_offload()` + VAE tiling keep peak VRAM low enough for a free-tier T4.

**Why this cell is slow:** the download is ~17GB total, and most of that isn't the video model
itself (the 1.3B-parameter transformer is only ~2.6GB) — it's the **UMT5-XXL text encoder**
(~5.7B parameters) that Wan uses to read the prompt. That's a one-time cost per Colab session
(downloads are already fast by default via Hugging Face's "Xet" transfer, no extra setup needed).
If you plan to come back to this notebook across multiple sessions, run the optional
Google-Drive-caching cell first so you only pay this cost once, ever.


**Optional — cache on Google Drive.** Two separate things get cached here, because they're
very different sizes:

1. **The quantized text encoder (~2-3GB)** — after the first run, the 4-bit-quantized UMT5-XXL
   gets saved back to Drive, so every future run loads that small file instead of re-downloading
   the ~21GB full-precision original (the entire 22.7GB you saw before was this one component —
   Wan's own repo doesn't host a smaller precision variant, so this is the only way to avoid
   paying for it more than once). This only needs ~5GB free and is worth doing almost always.
2. **The raw Hugging Face cache (~22-27GB, everything downloaded at full precision)** — optional,
   needs ~25GB free. Skip this if your Drive is tight; it only saves you from re-downloading the
   transformer+VAE (~5.7GB) on top of the text encoder, which is comparatively small anyway.

Skip this whole cell if you don't want to mount Drive at all.

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

QUANT_TE_DIR = "/content/drive/MyDrive/wan2.1_text_encoder_4bit"  # small (~2-3GB) — check this first
DRIVE_CACHE_DIR = "/content/drive/MyDrive/wan2.1_model_cache"      # large (~22-27GB) — optional

QUANT_TE_REQUIRED_GB = 5
FULL_CACHE_REQUIRED_GB = 25

free_gb = shutil.disk_usage("/content/drive/MyDrive").free / (1024 ** 3)
print(f"Google Drive free space: {free_gb:.1f} GB")

if free_gb >= QUANT_TE_REQUIRED_GB:
    os.makedirs(QUANT_TE_DIR, exist_ok=True)
    print("Quantized text-encoder cache enabled at:", QUANT_TE_DIR)
else:
    QUANT_TE_DIR = None
    print(f"Only {free_gb:.1f} GB free — not even enough for the small quantized-encoder "
          f"cache ({QUANT_TE_REQUIRED_GB} GB). It will be re-downloaded and re-quantized every session.")

if free_gb >= FULL_CACHE_REQUIRED_GB:
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ["HF_HOME"] = DRIVE_CACHE_DIR
    print("Full raw Hugging Face cache also redirected to Drive:", DRIVE_CACHE_DIR)
else:
    os.environ.pop("HF_HOME", None)  # use huggingface_hub's default local (non-persistent) cache
    print(f"Not enough free space for the full {FULL_CACHE_REQUIRED_GB}GB raw cache — "
          "transformer/VAE will use the Colab VM's local disk instead (fine, they're small).")


**About the "session crashed after using all available RAM" error:** that's *system* RAM,
not GPU VRAM — free Colab gives ~12-13GB of it, and Wan2.1's UMT5-XXL text encoder alone is
~5.7B parameters, which sits fully in CPU RAM under `enable_model_cpu_offload()`. That one
component can be close to the entire RAM budget by itself. The cell below loads *just the text
encoder* in 4-bit (via bitsandbytes) to shrink it roughly 4x, which is what actually fixes the
crash — the video transformer itself was never the problem.

In [ ]:
import os
import gc

# hf_transfer is deprecated (huggingface_hub now uses "Xet" for fast transfer by default,
# no env var needed) — nothing to set here anymore, downloads are already fast out of the box.

import torch
from transformers import UMT5EncoderModel, BitsAndBytesConfig
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
WAN_W, WAN_H = 640, 368
WAN_FPS = 16

# QUANT_TE_DIR is set by the optional Drive-caching cell above; define it here too in case
# that cell was skipped, so this cell still runs standalone (just without persistence).
if "QUANT_TE_DIR" not in dir():
    QUANT_TE_DIR = None

quant_te_cached = QUANT_TE_DIR and os.path.exists(os.path.join(QUANT_TE_DIR, "config.json"))

if quant_te_cached:
    print("Loading previously-quantized text encoder from Drive cache (fast, ~2-3GB, no big re-download)...")
    text_encoder = UMT5EncoderModel.from_pretrained(
        QUANT_TE_DIR, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
    )
else:
    print("No cached quantized text encoder found — downloading full-precision UMT5-XXL once (~21GB)...")
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,
    )
    text_encoder = UMT5EncoderModel.from_pretrained(
        MODEL_ID, subfolder="text_encoder", quantization_config=quant_config, torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    if QUANT_TE_DIR:
        os.makedirs(QUANT_TE_DIR, exist_ok=True)
        text_encoder.save_pretrained(QUANT_TE_DIR)
        print("Saved quantized text encoder to Drive for next time:", QUANT_TE_DIR)

print("Loading VAE + transformer...")
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(
    MODEL_ID, vae=vae, text_encoder=text_encoder, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
gc.collect()
torch.cuda.empty_cache()

NEGATIVE_PROMPT = (
    "blurry, low quality, distorted, deformed, extra limbs, bad anatomy, "
    "static image, watermark, text, subtitles, worst quality, jpeg artifacts, ugly"
)

print("model loaded:", MODEL_ID)


## 5. Generate an AI video clip per beat (Video Rendering agent)

Each clip is generated short (25-41 frames, ~1.5-2.5s) and gets looped/trimmed in Assembly to
fill its beat's actual narration length — this keeps generation time bounded regardless of how
long a stanza's narration runs. Expect a few minutes per clip on a T4; ~9 clips total.


In [ ]:
import os

os.makedirs("clips", exist_ok=True)
CLIP_PATHS = {}  # (beat_id, clip_index) -> mp4 path


def generate_with_retry(prompt, num_frames, width=WAN_W, height=WAN_H, steps=30, attempt=0):
    """640x368 already has a large safety margin under a T4's ~15GB, but GPUs get shared /
    fragmented unpredictably on free tiers — if it OOMs anyway, halve resolution once and retry
    rather than losing the whole session."""
    try:
        output = pipe(
            prompt=prompt, negative_prompt=NEGATIVE_PROMPT,
            height=height, width=width, num_frames=num_frames,
            guidance_scale=5.0, num_inference_steps=steps,
        )
        return output.frames[0]
    except torch.cuda.OutOfMemoryError:
        gc.collect()
        torch.cuda.empty_cache()
        if attempt >= 1:
            raise
        half_w = max(16, (width // 2) // 16 * 16)
        half_h = max(16, (height // 2) // 16 * 16)
        print(f"  OOM at {width}x{height} — retrying once at {half_w}x{half_h}...")
        return generate_with_retry(prompt, num_frames, half_w, half_h, steps, attempt + 1)


for beat in STORY["beats"]:
    for i, shot in enumerate(beat["clips"]):
        out_path = f"clips/{beat['id']}_{i}.mp4"
        print("generating:", beat["id"], i, "->", shot["prompt"])
        video_frames = generate_with_retry(shot["prompt"], shot["frames"])
        export_to_video(video_frames, out_path, fps=WAN_FPS)
        CLIP_PATHS[(beat["id"], i)] = out_path

        del video_frames
        gc.collect()
        torch.cuda.empty_cache()

print("done:", len(CLIP_PATHS), "clips")


## 6. Narration audio (Audio & Sync stage — Edge TTS, same voice as the static-image version)

In [ ]:
import asyncio
import edge_tts

os.makedirs("audio", exist_ok=True)
AUDIO_PATHS = {}

async def synth_all():
    for beat in STORY["beats"]:
        mp3_path = f"audio/{beat['id']}.mp3"
        communicate = edge_tts.Communicate(beat["narration"], STORY["voice"], rate="-8%")
        await communicate.save(mp3_path)
        AUDIO_PATHS[beat["id"]] = mp3_path
        print("synthesized:", beat["id"])

await synth_all()


## 7. Assembly (Video Rendering + Publishing agents)

For each beat: splits its narration duration across its `clips` by `hold` fraction (this is how
`stanza3`'s single audio track gets the monkey clip, then the fox clip, back to back), loops/trims
each AI clip to fill its share, letterboxes to 1280x720, overlays the Tamil subtitle bar (and a
title banner on intro/outro), concatenates every beat, muxes in the narration audio, and writes a
streaming-ready MP4 + SRT — same shape as the design doc's `Video Rendering` + `Publishing` agents.


In [ ]:
from moviepy import (
    AudioFileClip, VideoFileClip, VideoClip, concatenate_videoclips, concatenate_audioclips,
)

FPS = 24


def fit_letterbox(get_frame_fn, src_w, src_h):
    scale = min(TARGET_W / src_w, TARGET_H / src_h)
    new_w, new_h = int(src_w * scale), int(src_h * scale)
    x_off, y_off = (TARGET_W - new_w) // 2, (TARGET_H - new_h) // 2

    def make_frame(t):
        from PIL import Image as PILImage
        frame = PILImage.fromarray(get_frame_fn(t)).resize((new_w, new_h), PILImage.LANCZOS)
        canvas = np.zeros((TARGET_H, TARGET_W, 3), dtype=np.uint8)
        canvas[y_off:y_off + new_h, x_off:x_off + new_w] = np.array(frame)
        return canvas

    return make_frame


def looped_clip(path, duration):
    clip = VideoFileClip(path)
    src_w, src_h = clip.size
    frame_fn = fit_letterbox(lambda t: clip.get_frame(t % clip.duration), src_w, src_h)
    return VideoClip(frame_fn, duration=duration)


def overlay_rgba(base_clip, rgba_img, duration):
    arr = np.array(rgba_img)
    rgb = arr[:, :, :3].astype(np.float32)
    alpha = (arr[:, :, 3:4].astype(np.float32)) / 255.0

    def make_frame(t):
        under = base_clip.get_frame(t).astype(np.float32)
        comp = rgb * alpha + under * (1 - alpha)
        return comp.astype(np.uint8)

    return VideoClip(make_frame, duration=duration)


def srt_timestamp(seconds):
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


audio_clips_ordered = []
video_clips_ordered = []
srt_entries = []
t_cursor = 0.0

for beat in STORY["beats"]:
    bid = beat["id"]
    mp3_path = AUDIO_PATHS.get(bid, f"audio/{bid}.mp3")
    audio_clip = AudioFileClip(mp3_path)
    dur = audio_clip.duration
    audio_clips_ordered.append(audio_clip)

    hold_sum = sum(shot["hold"] for shot in beat["clips"])
    sub_clips = []
    for i, shot in enumerate(beat["clips"]):
        shot_dur = dur * (shot["hold"] / hold_sum)
        clip_path = CLIP_PATHS.get((bid, i), f"clips/{bid}_{i}.mp4")
        sub_clips.append(looped_clip(clip_path, shot_dur))
    visual = concatenate_videoclips(sub_clips) if len(sub_clips) > 1 else sub_clips[0]

    cap = caption_bar(beat["narration"])
    visual = overlay_rgba(visual, cap, dur)
    if bid in ("intro", "outro"):
        banner = title_banner(STORY["chapter_title"])
        visual = overlay_rgba(visual, banner, dur)

    video_clips_ordered.append(visual)
    srt_entries.append((t_cursor, t_cursor + dur, beat["narration"]))
    t_cursor += dur

final_video = concatenate_videoclips(video_clips_ordered, method="compose")
final_audio = concatenate_audioclips(audio_clips_ordered)
final_video = final_video.with_audio(final_audio)

os.makedirs("output", exist_ok=True)
mp4_path = "output/aalamaram_vilaiyaatu_ai.mp4"
final_video.write_videofile(
    mp4_path, fps=FPS, codec="libx264", audio_codec="aac", preset="medium",
    ffmpeg_params=["-movflags", "+faststart", "-crf", "20"],
)

srt_path = "output/aalamaram_vilaiyaatu_ai.srt"
with open(srt_path, "w", encoding="utf-8") as f:
    for i, (start, end, text) in enumerate(srt_entries, 1):
        f.write(f"{i}\n{srt_timestamp(start)} --> {srt_timestamp(end)}\n{text}\n\n")

print("Video:", mp4_path)
print("Subtitles:", srt_path)


## 8. Download the result

In [ ]:
from google.colab import files
files.download(mp4_path)
files.download(srt_path)
